# 07 · Validación de ERA5-Land con estaciones de INAMHI

**Fase CRISP-DM:** comprensión de los datos (verificación de calidad).

**Objetivo:** comprobar si ERA5-Land, la fuente climática de nivel 1, representa bien el clima de Loja y Zamora Chinchipe, comparándola con las estaciones meteorológicas de INAMHI (nivel 3 del documento de fuentes v4). Responde tres preguntas:

1. ¿ERA5-Land tiene un **sesgo sistemático** (da más lluvia o más calor que las estaciones)?
2. ¿ERA5-Land **sube y baja junto con las estaciones**? Es lo que usa el modelo: las anomalías frente a lo normal.
3. ¿Hace falta **corregir** ERA5-Land antes de usarla en la configuración B?

**Período:** 1991-2019, donde se superponen las dos fuentes (la entrega de INAMHI termina en diciembre de 2019).

**Datos:** públicos, sin PII. Las estaciones están en `referencias/inamhi_*.csv` (ver `scripts/inamhi_estaciones.py`) y ERA5-Land se lee de `raw.era5_land` del lakehouse de la plataforma (`LOJAGAS_LAKEHOUSE`, por defecto `~/lojagas_datos/lakehouse`). Antes de correr este notebook, la ingesta externa tiene que haber escrito esa tabla.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import val_inamhi as vi

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)

## 1. Estaciones y limpieza

Se descartan los valores fuera de un rango físico posible para Loja y Zamora (`vi.RANGOS`): son errores de digitación, como un mes con 5544,9 mm. La tabla muestra cuáles se descartaron, para revisarlos.

In [ ]:
estaciones, largo = vi.cargar_estaciones()
valores, descartados = vi.filtrar_valores(largo)

print(f"Estaciones: {len(estaciones)} ({estaciones.groupby('provincia').size().to_dict()})")
desde, hasta = vi.PERIODO
print(f"Valores {desde}-{hasta}: {len(valores):,} válidos, {len(descartados)} descartados")
descartados.merge(estaciones[["estacion", "nombre"]], on="estacion")

## 2. ERA5-Land y el píxel más cercano a cada estación

Cada estación se compara con el centro de píxel de ERA5-Land más cercano (malla de 0,1°, unos 11 km de lado). Una distancia mayor a unos 8 km indica que la estación está cerca del borde de la zona descargada.

In [ ]:
pixeles = vi.era5_por_pixel()
cercanos = vi.pixel_mas_cercano(estaciones, pixeles)

print(
    f"ERA5-Land: {pixeles[['lat', 'lon']].drop_duplicates().shape[0]} píxeles, "
    f"{pixeles['mes'].min():%Y-%m} a {pixeles['mes'].max():%Y-%m}"
)
cercanos["distancia_km"].describe().round(1)

## 3. Métricas por estación

Solo estaciones con al menos `vi.MIN_MESES` meses comparables (10 años).

- **sesgo**: ERA5 − estación, en mm/mes o °C. Positivo si ERA5-Land da de más.
- **sesgo_pct**: el mismo sesgo como % de lo que midió la estación. Útil para la lluvia.
- **rmse**: error típico de un mes cualquiera.
- **r**: correlación de las series tal cual. Incluye el ciclo anual, así que suele salir alta.
- **r_anomalias**: correlación de las anomalías (cada mes menos su promedio de ese mes del año). Es la exigente: dice si ERA5-Land detecta que un mes fue más lluvioso o más frío de lo normal.

In [ ]:
pares = vi.con_anomalias(vi.emparejar(valores, pixeles, cercanos))
metricas = vi.metricas_por_estacion(pares)
metricas.merge(estaciones[["estacion", "nombre"]], on="estacion").round(2)

## 4. Resumen por variable y provincia

Mediana entre estaciones, para que una estación con problemas no mueva el resultado.

In [ ]:
vi.resumen(metricas).round(2)

## 5. ¿El sesgo de temperatura es por la altitud?

Un píxel de ERA5-Land promedia unos 11 km de terreno montañoso; una estación en un valle o en una cumbre está a otra altura que su píxel. Si el sesgo de temperatura se debe a eso, crece con la altitud de la estación a un ritmo cercano al gradiente térmico (~6,5 °C por km). En ese caso no es un error de ERA5-Land sino de representatividad, y no afecta a las anomalías que usa el modelo.

In [ ]:
print(vi.sesgo_por_altitud(metricas))

t = metricas[metricas["variable"] == "temp_media_c"]
fig, ax = plt.subplots(figsize=(6, 4))
for provincia, g in t.groupby("provincia"):
    ax.scatter(g["altitud_m"], g["sesgo"], label=provincia)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("Altitud de la estación (m)")
ax.set_ylabel("Sesgo ERA5 − estación (°C)")
ax.set_title("Sesgo de temperatura según la altitud")
ax.legend()
plt.show()

## 6. Una estación de cerca: La Argelia (M0033)

Es la estación con la serie más completa de Loja (casi sin huecos entre 1991 y 2019), en la ciudad de Loja.

In [ ]:
argelia = pares[pares["estacion"] == "M0033"]
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for ax, (variable, g) in zip(axes, argelia.groupby("variable"), strict=False):
    ax.plot(g["mes"], g["estacion_valor"], label="INAMHI", lw=1)
    ax.plot(g["mes"], g["era5"], label="ERA5-Land", lw=1)
    ax.set_ylabel(variable)
    ax.legend(loc="upper right")
axes[0].set_title("La Argelia (M0033): estación frente a su píxel de ERA5-Land")
plt.tight_layout()
plt.show()

## 7. Ciclo anual por provincia

Promedio de cada mes del año entre las estaciones con métricas. Si ERA5-Land reproduce la forma del ciclo (meses lluviosos y secos, meses fríos), sirve para el modelo aunque tenga un sesgo constante.

In [ ]:
validas = pares[pares["estacion"].isin(metricas["estacion"])]
claves = ["variable", "provincia", "mes_del_anio"]
ciclo = validas.groupby(claves)[["estacion_valor", "era5"]].mean()

combinaciones = ciclo.index.droplevel("mes_del_anio").unique()
fig, axes = plt.subplots(1, len(combinaciones), figsize=(4 * len(combinaciones), 3.5))
for ax, (variable, provincia) in zip(axes, combinaciones, strict=False):
    g = ciclo.loc[(variable, provincia)]
    ax.plot(g.index, g["estacion_valor"], marker="o", label="INAMHI")
    ax.plot(g.index, g["era5"], marker="o", label="ERA5-Land")
    ax.set_title(f"{variable}\n{provincia}")
    ax.set_xticks(range(1, 13))
axes[0].legend()
plt.tight_layout()
plt.show()

## Decisiones

_Completar al ver los resultados. Las cifras quedan en las salidas de este notebook, que no se suben al repositorio; aquí van las conclusiones, en palabras._

Preguntas que tiene que responder esta sección:

1. **Sesgo de lluvia.** ¿ERA5-Land da sistemáticamente más o menos lluvia? ¿Cuánto, en %, y es parecido en Loja y en Zamora?
2. **Sesgo de temperatura.** ¿Se explica por la altitud (pendiente cercana a 6,5 °C/km en la sección 5)? Si es así, no es un error de ERA5-Land.
3. **Anomalías.** ¿`r_anomalias` es alto? Como referencia, por encima de 0,7 ERA5-Land detecta bien los meses anómalos; por debajo de 0,5, las variables de anomalía del modelo serían poco confiables.
4. **¿Corregir?** El modelo usa anomalías y modelos que estandarizan las variables: un sesgo constante no cambia sus predicciones. Una corrección solo haría falta si el sesgo cambia según el mes del año (sección 7) o si se quieren reportar valores absolutos.
5. **Limitaciones.** La temperatura de Zamora Chinchipe tiene una sola estación con serie parcial (Yanzatza); la comparación es de punto (estación) contra área (píxel).

## Decisiones

1. **Lluvia en Loja: ERA5-Land da más lluvia que las estaciones (mediana +54 %), en forma proporcional.** El exceso aparece en todos los meses y es mayor en la época seca, donde llega a más del doble: es un sesgo multiplicativo. Aun así, sigue los meses anómalos de forma moderada (r de anomalías 0,58), y mejor en las estaciones bajas del oeste (0,72-0,80) que en la sierra.
2. **Lluvia en Zamora Chinchipe: ERA5-Land no la representa.** El sesgo medio es casi nulo, pero el ciclo anual está desplazado: ERA5-Land ubica el pico en febrero-marzo y las estaciones en abril-julio (régimen amazónico). La correlación es baja (r 0,35; r de anomalías 0,30). No se usa en la configuración B.
3. **Temperatura: ERA5-Land es 2-4 °C más fría, con la forma del ciclo anual correcta.** El sesgo no se explica por la altitud de la estación (pendiente 1,1 °C/km, R² 0,05, frente a 6,5 esperados), aunque la prueba es débil porque no usa la altitud del píxel. Las anomalías coinciden de forma moderada en Loja (0,48) y bien en la única estación útil de Zamora (0,76).
4. **No se corrige ERA5-Land para el modelo,** que usa anomalías y variables estandarizadas. Pero la anomalía de lluvia debe ser relativa (lluvia del mes / normal del mes), no una resta en mm, para que el sesgo multiplicativo se cancele. Los valores absolutos de ERA5-Land no se reportan como reales.
5. **Limitaciones.** Es una comparación de punto (estación) contra área (píxel de ~11 km). La temperatura de Zamora tiene una sola estación con serie parcial (Yanzatza). Tres estaciones de Loja (Buenavista, Chaguarpamba y El Prado) se excluyen: su píxel de ERA5-Land da valores irreales (más de 4000 mm al año) y El Prado tiene además errores de digitación.